

---


# **Subject : Automatically categorize questions on Stackoverflow**





---











To ask a question un StackOverflow, people need to select some tags in order to maximize their chance to get an answer. In this project, we are volunteer to help StackOverflow by developping an algorithm that will suggest tags based on the title and/or the content of the question.

For doing this, we will need first get some data for training and testing our algorithms. StackOverflow offers a data export tool, "[StackExchange Data Explorer](https://data.stackexchange.com/stackoverflow/query/new)", which lists a large amount of authentic data from the platform.
Therefore you can run the following query in this data export tool to create your dataset and download it:

```
SELECT TOP 10000 Title, Body, Tags, Score, ViewCount, FavoriteCount, AnswerCount, CreationDate
FROM Posts
WHERE ViewCount > 10
AND AnswerCount > 5
AND LEN(Tags) - LEN(REPLACE(Tags, '<','')) <= 4
AND LEN(Tags) - LEN(REPLACE(Tags, '<','')) >= 1
```

If you can't download the data by yourself, you can use the files available in this link : [here](https://1drv.ms/f/c/ab584826bfa6bf60/EmC_pr8mSFgggKuwDgAAAAABUQ7Gb55TD9Sc_u27Dezm2A?e=0Tw5fz)


Below are the steps required to fullful this project :



1.   Process the "Tags" column in order to compute some statistics : wordcloud on the tags, bar diagram with the 20 most common tags
2.   Filter the dataset to keep only rows containing the top 10 tags
3.   Combine the title and the body of the question for the analysis. Apply cleaning process on the corpus (title+body)
4.   Implement an unsupervised approach (LDA for example) to identify main topics/key word on the dataset. Propose some graphics to illustrate the results
5.   Implement a supervised approach to predict the tags (use [this first example](https://www.kaggle.com/discussions/questions-and-answers/66693) to process the target into multiple binary outputs and [this second example](https://dongr0510.medium.com/multi-label-classification-example-with-multioutputclassifier-and-xgboost-in-python-98c84c7d379f) to know how to predicting a multilabel target). Make sure to test and compare the following feature extraction methods :

    *   1 bag-of-word approach : TF-IDF or CountVectorizer
    *   2 word embedding approach among these : Word2Vec, Glove, BERT, USE

  For the word embedding part, you can use [this example of notebook](https://s3.eu-west-1.amazonaws.com/course.oc-static.com/projects/Data_Scientist_P6/Exemple_Tweets_Feature-extraction_Sentence+Embedding_V1.1.ipynb) to help you explore Word2Vec, Doc2Vec, Glove, BERT et USE.

  For the supervised task, test 2 or more models and tune hyperparameters if possible.

6.   Evaluate and compare the models trained after using a train/test split


In [ ]:
# Imports

In [ ]:
# Packages needed for this notebook — uncomment and run once on a fresh environment (e.g. personal machine / Google Colab)
# %pip install pandas numpy matplotlib beautifulsoup4 nltk wordcloud scikit-learn gensim sentence-transformers xgboost

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import pandas as pd
import numpy as np
import re
import nltk
from bs4 import BeautifulSoup
nltk.download('punkt')
nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))



In [2]:
df = pd.read_csv('QueryResults.csv')

In [ ]:
df.head()

### 1. **Processing "Tags"**

In [ ]:
## Transform tags values into list of tags
def process_tag(s):
  return(list(filter(None, re.split(r'<|>' , s))))

df['tag_list'] =  df.Tags.apply(process_tag)
df[['Tags', 'tag_list']].head()

In [ ]:
# Get list of all tags
l_tags = df.tag_list.apply(lambda x : " ".join(x)).tolist()
all_tags = " ".join(l_tags)
all_tags

In [16]:
# List of unique tags
unique_tags = np.unique(all_tags.split(' '))
len(unique_tags)

4000

In [ ]:
# Count of occurence of tags
(pd.DataFrame(all_tags.split(' '), columns = ['tag'])\
  .groupby('tag').agg({'tag' : 'count'})\
  .rename(columns={'tag': 'count'})\
  .reset_index()\
  .sort_values('count', ascending = False)\
  .head(15))

In [ ]:
## TODO :=> wordcloud on the tags, bar diagram with the 20 most common tags
from wordcloud import WordCloud
from collections import Counter

tag_counts = Counter(all_tags.split(' '))

# Wordcloud on the tags
wc = WordCloud(width=1200, height=600, background_color='white', colormap='viridis')\
      .generate_from_frequencies(tag_counts)
plt.figure(figsize=(14, 7))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('Wordcloud of StackOverflow tags')
plt.show()

# Bar diagram with the 20 most common tags
top20 = pd.DataFrame(tag_counts.most_common(20), columns=['tag', 'count'])
plt.figure(figsize=(12, 6))
plt.bar(top20['tag'], top20['count'], color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.xlabel('Tag')
plt.ylabel('Number of occurrences')
plt.title('20 most common tags')
plt.tight_layout()
plt.show()

### 2. **Filter dataset**

In [ ]:
## TODO :=> Take the real list of 10 most frequent tags
most_frequent_tags = [tag for tag, count in Counter(all_tags.split(' ')).most_common(10)]
print("Top 10 tags :", most_frequent_tags)

def intersection(lst1, lst2):
    return list(set(lst1) & set(lst2))

def keep_indicator(l_tags):
  intersect = intersection(l_tags, most_frequent_tags)
  return(int(len(intersect) > 0))

keep_indicator(['tag', "c"])

In [ ]:
# Create indicator for keeping row or not
df['keep_indicator'] = df['tag_list'].apply(keep_indicator)
df[['tag_list', 'keep_indicator']].head()

In [ ]:
# Clean the tag_list for keeping only selected tags => this cleaned column will be used for modeling
def simplify_tag_list(l):
  return([x for x in l if x in most_frequent_tags])

df['tag_list_final'] = df['tag_list'].apply(simplify_tag_list)
df['n_tags'] = df['tag_list_final'].apply(lambda x : len(x))
df[['tag_list', 'keep_indicator', 'tag_list_final', 'n_tags']].sort_values('n_tags', ascending = False).head(5)

In [ ]:
## TODO :=> Filter the dataset to keep only rows with selected tags
print("Shape before filtering :", df.shape)
df = df[df['keep_indicator'] == 1].reset_index(drop=True)
print("Shape after filtering  :", df.shape)
df['n_tags'].value_counts()

### 3. **Cleaning process**

In [ ]:
# TODO :=> complete the cleaning function to add classical cleaning process in NLP (prefer lemmatizing to stemming :)  )
# Concatenate Title & body, then clean
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

def remove_tags(text):
  soup = BeautifulSoup(text, "html.parser")
  for data in soup(['style', 'script']):
    data.decompose()
  return ' '.join(soup.stripped_strings)

def clean_text(text):
  s = text.lower() # Lowercase
  s = remove_tags(s) # remove html tags "<.>"
  s = re.sub('\n','', s) # remove line breaks
  s = re.sub(r'(&lt;)|(&gt;)','', s) # remove special sequences
  #... => other cleaning operations
  s = re.sub(r'http\S+|www\.\S+', ' ', s) # remove URLs
  s = re.sub(r'[^a-z#+\s]', ' ', s) # keep only letters ('#' and '+' kept to preserve tags like c# / c++)
  tokens = s.split() # tokenization
  tokens = [w for w in tokens if w not in stop_words] # remove stopwords
  tokens = [w for w in tokens if len(w) > 1] # remove single characters
  tokens = [lemmatizer.lemmatize(w) for w in tokens] # lemmatization (preferred to stemming)
  s = " ".join(tokens)
  return(s)


df['text'] = df[['Title', 'Body']].apply(" ".join, axis=1).apply(clean_text)
# Light version (only html removed) kept for sentence embeddings later (BERT works better on full sentences)
df['text_light'] = df[['Title', 'Body']].apply(" ".join, axis=1).apply(remove_tags)
df[['Title', 'Body', 'text']].loc[0,:].tolist()

### 4. **Unsupervised approach**

In [ ]:
# TODO :=> apply an unsupervised approach
## LDA (Latent Dirichlet Allocation) to identify the main topics of the corpus
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

count_vect = CountVectorizer(max_features=5000, max_df=0.95, min_df=5)
X_counts = count_vect.fit_transform(df['text'])
print("Document-term matrix :", X_counts.shape)

n_topics = 10
lda = LatentDirichletAllocation(n_components=n_topics, max_iter=10,
                                learning_method='online', random_state=42)
doc_topics = lda.fit_transform(X_counts)

# Top words per topic
feature_names = count_vect.get_feature_names_out()
for topic_idx, topic in enumerate(lda.components_):
    top_words = [feature_names[i] for i in topic.argsort()[:-11:-1]]
    print(f"Topic {topic_idx} : {' | '.join(top_words)}")

In [ ]:
## Graphics to illustrate the LDA results

# 1. Top 10 words per topic (bar charts)
fig, axes = plt.subplots(2, 5, figsize=(22, 8))
for topic_idx, (topic, ax) in enumerate(zip(lda.components_, axes.flatten())):
    top_idx = topic.argsort()[:-11:-1]
    ax.barh([feature_names[i] for i in top_idx][::-1], topic[top_idx][::-1], color='steelblue')
    ax.set_title(f"Topic {topic_idx}")
plt.suptitle("LDA - Top 10 words per topic", fontsize=14)
plt.tight_layout()
plt.show()

# 2. Link between dominant LDA topic and the real tags
df['dominant_topic'] = doc_topics.argmax(axis=1)
df['main_tag'] = df['tag_list_final'].apply(lambda l: l[0] if len(l) > 0 else None)
ct = pd.crosstab(df['dominant_topic'], df['main_tag'])

plt.figure(figsize=(10, 6))
plt.imshow(ct, aspect='auto', cmap='Blues')
plt.colorbar(label='Number of questions')
plt.xticks(range(len(ct.columns)), ct.columns, rotation=45, ha='right')
plt.yticks(range(len(ct.index)), ct.index)
plt.xlabel('Main tag')
plt.ylabel('Dominant topic (LDA)')
plt.title('Distribution of tags across LDA topics')
plt.tight_layout()
plt.show()

### 5. **Supervised approach**

In [ ]:
# TODO :=> apply word embeddings and supervised approaches

## --- Target preparation : multi-label binarization + train/test split ---
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split

mlb = MultiLabelBinarizer(classes=most_frequent_tags)
y = mlb.fit_transform(df['tag_list_final'])
print("Target shape :", y.shape, "| labels :", list(mlb.classes_))

idx_train, idx_test = train_test_split(np.arange(len(df)), test_size=0.2, random_state=42)
y_train, y_test = y[idx_train], y[idx_test]
print("Train :", len(idx_train), "| Test :", len(idx_test))

## --- Feature extraction 1 : Bag-of-words (TF-IDF) ---
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(df['text'].iloc[idx_train])
X_test_tfidf  = tfidf.transform(df['text'].iloc[idx_test])
print("TF-IDF features :", X_train_tfidf.shape)

In [ ]:
## --- Feature extraction 2 : Word2Vec (average of word vectors per document) ---
from gensim.models import Word2Vec

tokens_train = [t.split() for t in df['text'].iloc[idx_train]]
tokens_test  = [t.split() for t in df['text'].iloc[idx_test]]

w2v = Word2Vec(sentences=tokens_train, vector_size=300, window=5,
               min_count=3, workers=4, seed=42, epochs=20)

def doc_vector(tokens, model):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    return np.mean(vecs, axis=0) if len(vecs) > 0 else np.zeros(model.vector_size)

X_train_w2v = np.vstack([doc_vector(t, w2v) for t in tokens_train])
X_test_w2v  = np.vstack([doc_vector(t, w2v) for t in tokens_test])
print("Word2Vec features :", X_train_w2v.shape)

## --- Feature extraction 3 : BERT sentence embeddings ---
from sentence_transformers import SentenceTransformer

bert = SentenceTransformer('all-MiniLM-L6-v2')
X_train_bert = bert.encode(df['text_light'].iloc[idx_train].tolist(), show_progress_bar=True)
X_test_bert  = bert.encode(df['text_light'].iloc[idx_test].tolist(), show_progress_bar=True)
print("BERT features :", X_train_bert.shape)

## --- Supervised models : Logistic Regression (OneVsRest) & XGBoost (MultiOutputClassifier) ---
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multioutput import MultiOutputClassifier
from xgboost import XGBClassifier

features = {
    'TF-IDF'  : (X_train_tfidf, X_test_tfidf),
    'Word2Vec': (X_train_w2v, X_test_w2v),
    'BERT'    : (X_train_bert, X_test_bert),
}

def make_models():
    return {
        'LogisticRegression': OneVsRestClassifier(LogisticRegression(max_iter=1000)),
        'XGBoost': MultiOutputClassifier(XGBClassifier(n_estimators=200, max_depth=6,
                                                       learning_rate=0.1, eval_metric='logloss')),
    }

trained = {}
for feat_name, (X_tr, X_te) in features.items():
    for model_name, model in make_models().items():
        print(f"Training {model_name} on {feat_name} ...")
        model.fit(X_tr, y_train)
        trained[(feat_name, model_name)] = model
print("Done :", len(trained), "models trained")

## --- Hyperparameter tuning (example : TF-IDF + Logistic Regression) ---
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(OneVsRestClassifier(LogisticRegression(max_iter=1000)),
                    param_grid={'estimator__C': [0.1, 1, 10]},
                    scoring='f1_micro', cv=3, n_jobs=-1)
grid.fit(X_train_tfidf, y_train)
print("Best params :", grid.best_params_, "| best CV f1_micro :", round(grid.best_score_, 3))
trained[('TF-IDF', 'LogisticRegression')] = grid.best_estimator_

### 6. **Evaluation/Comparison**

In [ ]:
# TODO :=> Evaluate & compare models
from sklearn.metrics import f1_score, hamming_loss, jaccard_score, accuracy_score, classification_report

results = []
for (feat_name, model_name), model in trained.items():
    y_pred = model.predict(features[feat_name][1])
    results.append({
        'features'        : feat_name,
        'model'           : model_name,
        'f1_micro'        : f1_score(y_test, y_pred, average='micro'),
        'f1_macro'        : f1_score(y_test, y_pred, average='macro'),
        'jaccard'         : jaccard_score(y_test, y_pred, average='samples', zero_division=0),
        'hamming_loss'    : hamming_loss(y_test, y_pred),
        'subset_accuracy' : accuracy_score(y_test, y_pred),
    })

results_df = pd.DataFrame(results).sort_values('f1_micro', ascending=False).reset_index(drop=True)
results_df.round(3)

In [ ]:
# Comparison chart + detailed report of the best model
pivot = results_df.pivot(index='features', columns='model', values='f1_micro')
pivot.plot(kind='bar', figsize=(10, 5), rot=0)
plt.ylabel('F1-score (micro)')
plt.title('Model comparison : features x classifier (test set)')
plt.legend(title='Model')
plt.tight_layout()
plt.show()

best = results_df.iloc[0]
print(f"Best combination : {best['features']} + {best['model']} (f1_micro = {best['f1_micro']:.3f})\n")
y_pred_best = trained[(best['features'], best['model'])].predict(features[best['features']][1])
print(classification_report(y_test, y_pred_best, target_names=mlb.classes_, zero_division=0))